# Create Subsets for Evaluation

## Import packages

In [ ]:
from pyspark.sql import SparkSession
import numpy as np

## Set parameters

In [ ]:
data_path = "catalog.schema.table"
data = spark.sql("""select * from """ + data_path)
target_col = "MY_TARGET"
access_group = "group-name"

In [ ]:
# N Longest Records with X Unique Start Words parameters
N = 1000
X = 3
storage_container_path_longest = "abfss://container/folder/file.csv"
output_table_name_longest = "catalog.schema.output_table"

In [ ]:
# M Random Records from each Reporting Unit parameters
M = 50
unit_col = "MY_UNIT_COL"
storage_container_path_random = ""
output_table_name_random = "catalog.schema.output_table"

## _N_ Longest Records with _X_ Unique Start Words

In [ ]:
sorted_results_df = spark.sql(
    "SELECT * FROM {df} ORDER BY LENGTH(TRIM(" + target_col + ")) DESC",
    df=data,
).toPandas()

In [ ]:
existing_start_words = []


def flag_unique_start_rows(target_val):

    start_words = target_val.split()[:X]
    start_words = " ".join(start_words)

    if start_words not in existing_start_words:
        existing_start_words.append(start_words)
        return 1
    else:
        return 0

In [ ]:
unique_start_flag = []
existing_start_words = []

for idx, row in sorted_results_df.iterrows():
    val = flag_unique_start_rows(row[target_col])
    unique_start_flag.append(val)
    if sum(unique_start_flag) >= N:  # Your "until" logic
        break

In [ ]:
target_len = len(sorted_results_df)
unique_start_flag = unique_start_flag + [np.nan] * (
    target_len - len(unique_start_flag)
)

In [ ]:
sorted_results_df["unique_start_flag"] = unique_start_flag
longest_unique_start_df = sorted_results_df[
    sorted_results_df["unique_start_flag"] == 1
].drop(columns=["unique_start_flag"])

In [ ]:
# Convert Pandas DataFrame to PySpark DataFrame
spark = SparkSession.builder.appName("PandasToSparkConversion").getOrCreate()

longest_spark_df = spark.createDataFrame(longest_unique_start_df)

In [ ]:
# Save to catalog
longest_spark_df.write.mode("overwrite").saveAsTable(output_table_name_longest)

In [ ]:
# Write it to blob storage as CSV
longest_spark_df.coalesce(1).write.format("csv").option("header", "true").mode(
    "overwrite"
).save(storage_container_path_longest)

## _M_ Random Records from each Reporting Unit

In [ ]:
data_df = data.toPandas()

# Remove random_state variable if you want to have different results each time
rand_strat_sample_df = data_df.groupby(unit_col, group_keys=False).apply(
    lambda x: x.sample(M, random_state=123)
)

In [ ]:
# Convert Pandas DataFrame to PySpark DataFrame
spark = SparkSession.builder.appName("PandasToSparkConversion").getOrCreate()

random_spark_df = spark.createDataFrame(rand_strat_sample_df)

In [ ]:
# Save to catalog
random_spark_df.write.mode("overwrite").saveAsTable(output_table_name_random)

In [ ]:
# Write it to blob storage as CSV
random_spark_df.coalesce(1).write.format("csv").option("header", "true").mode(
    "overwrite"
).save(storage_container_path_random)

## Create Other Subsets as Suitable for your Case

Grant permissions as needed

In [ ]:
spark.sql(
    "GRANT SELECT ON " + output_table_name_longest + " TO " + access_group
)
spark.sql(
    "GRANT SELECT ON " + output_table_name_random + " TO " + access_group
)